In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Patch
import numpy as np
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)


### **1. Data Overview & Integrity**

#### **1.1. Dataset Overview**

- Source: https://www.kaggle.com/datasets/muhammadshahidazeem/customer-churn-dataset/data
- Total number of observations (rows): 440,831
- Total number of attributes (columns): 11
- Target Variables: `Churn` (Binary class)
- Numerical Features: `Age`, `Tenure`, `Usage Frequency`, `Support Calls`, `Payment Delay`, `Total Spend`, `Last Interaction`
- Category Features: `Gender`, `Subscription Type`, `Contract Length`


| Variable Name | Description | Feature Taxonomy | Example (Value & Unit) |
| :--- | :--- | :--- | :--- |
| **Age** | The age of the customer. | Demographics | `28` (Years) |
| **Gender** | The biological sex of the customer. | Demographics | `Male` / `Female` |
| **Tenure** | Duration of the customer's relationship with the company. | Usage Behavior | `12` (Months) |
| **Usage Frequency** | How often the customer uses the service per month. | Usage Behavior | `15` (Times/Month) |
| **Support Calls** | The number of times the customer contacted customer support center. | Usage Behavior | `4` (Calls) |
| **Payment Delay** | Number of days the customer is late on their billing. | Usage Behavior | `7` (Days) |
| **Last Interaction** | Days elapsed since the last customer engagement. | Usage Behavior | `3` (Days ago) |
| **Subscription Type** | The specific service tier chosen by the customer. | Contractual Info | `Basic`, `Standard`, `Premium` |
| **Contract Length** | The commitment period of the service agreement. | Contractual Info | `Monthly`, `Quarterly`, `Annual` |
| **Total Spend** | Total revenue generated from the customer to date. | Contractual Info | `1250.50` (Currency Units) |
| **Churn (Target Variable)** | Target variable indicating if the customer has left. | Contractual Info | `1` (Yes) or `0` (No) |

*Note: The dataset used in this project is a **synthetic/generated dataset** from Kaggle. While it lacks some real-world noise, its large scale (440k+ records) allows for a robust demonstration of MLOps pipeline automation and model scalability.*

In [ ]:
# 1. Download data
df = pd.read_csv("../data/raw/train.csv")
df.head(3)

#### **1.2. Data Quality & Integrity Check**

**Data shape and Info**

In [ ]:
# Data shape
df.shape

In [ ]:
# Data basic Info
df.info()

**Missing Value and Duplication**

In [ ]:
# Missing value 

print(f"Total null value: {df.isnull().sum().sum()}\n")
print("Total missing value for each columns:")
df.isnull().sum()

In [ ]:
# Duplication
df = df.drop_duplicates()

# Drop CustomerID
df = df.drop(columns='CustomerID')
df.head(3)

**Data Type Consistency**

In [ ]:
# transform Churn into int 
df['Churn'] = df['Churn'].astype(int)

# transform Age into int 
df['Age'] = df['Age'].astype(int)

# Category
category_cols = ['Gender', 'Subscription Type', 'Contract Length']

for col in category_cols:
    df[col] = df[col].astype('str')

**Statistical Validity**

The dataset contains 440,832 observations with no missing, duplicate or unlogical values, reflecting a clean and complete structure. Numerical features show reasonable ranges and variability without extreme outliers, and the churn rate (~57%) indicates slight class imbalance.

As the data is synthetically generated, distributions appear well-behaved. In real-world scenarios, additional preprocessing steps such as handling missing values, removing duplicates, and performing sanity checks would be required to ensure data quality.

In [ ]:
# Numeric Variables
df.describe().round(2)

Categorical variables are relatively well-balanced (as expected from generated data), with Subscription Type evenly distributed and a mild imbalance observed in Gender. Contract Length is skewed toward Annual and Quarterly plans, with Monthly representing a smaller segment.

However, in preprocessing step, Categorical values were still standardized using simple formatting (title casing) to ensure consistency. Inconsistent or unseen categories are handled during encoding (e.g., ignored if not recognized).

In [ ]:
# Category Variables

gender = df['Gender'].value_counts()
sub_type = df['Subscription Type'].value_counts()
contract = df['Contract Length'].value_counts()

print(gender,'\n\n')
print(sub_type, '\n\n')
print(contract)

### **2. Target Variable and Metrics**

#### **2.1. Target Variable (Churn)**

The target variable Churn was analyzed for class distribution to determine if specialized sampling techniques were required.

Class Distribution is quite balanced with a distribution of 56.7% Churn (Yes) and 43.3% Non-Churn (No). 

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

sns.countplot(data=df, x="Churn", ax=ax, palette="pastel") 

plt.title("The distribution for Churn")

counts = df['Churn'].value_counts().sort_index()
pct = (counts / counts.sum() * 100).round(1)

colors = [p.get_facecolor() for p in ax.patches]

handles = [
    Patch(facecolor=colors[0], label=f'No (0): {pct.get(0, 0):.1f}%'),
    Patch(facecolor=colors[1], label=f'Yes (1): {pct.get(1, 0):.1f}%')
]

ax.legend(handles=handles, loc='upper right', bbox_to_anchor=(1.5, 1))
fig.subplots_adjust(right=0.75) 

plt.show()

#### **2.2. Metrics**

Although the generated dataset is relatively balanced (~50/50), **F1-score** is selected as the primary metric to ensure robustness under potential class imbalance in real-world scenarios, while additional metrics such as Precision, Recall, and Accuracy are also reported for comprehensive evaluation.

| Metric        | Formula | Reasons to Choose |
|--------------|--------|------------------|
| **F1-Score** | $$\text{F1} = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$$ | Balances Precision and Recall, making it suitable for imbalanced datasets like churn prediction where both false positives and false negatives matter. |
| **Precision** | $$\text{Precision} = \frac{TP}{TP + FP}$$ | Helps reduce unnecessary retention costs by ensuring that predicted churners are truly at risk. |
| **Recall** | $$\text{Recall} = \frac{TP}{TP + FN}$$ | Critical for churn prediction, as it ensures most actual churners are identified, minimizing customer loss. |
| **Accuracy** | $$\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}$$ | Provides an overall performance baseline but can be misleading in real-world imbalanced datasets. |
| **F1-Macro** | $$\text{F1-Macro} = \frac{1}{N} \sum_{i=1}^{N} \text{F1}_i$$ | Treats all classes equally, useful for detecting model bias toward majority classes. |
| **F1-Weighted** | $$\text{F1-Weighted} = \sum_{i=1}^{N} \left( \text{F1}_i \cdot \frac{\text{Support}_i}{\text{Total Samples}} \right)$$ | Reflects overall performance by weighting classes according to their frequency, suitable for imbalanced datasets. |


#### **3. Feature Distribution**

#### **3.1. Numerical Features**

##### **General Observations**
- **Uniformity:** Features like `Tenure`, `Usage Frequency`, and `Payment Delay` exhibit a near-uniform distribution across their ranges. This could be due to synthetic dataset.
- **No Outliers:** Based on the boxplots and the Interquartile Range (IQR), there are no significant statistical outliers in any numerical columns. All values fall within logical business bounds.
- **Symmetry:** Most features show a high degree of symmetry, with mean and median (50%) values being very close, indicating minimal skewness.

#### **Specific Feature Insights**
- **Age:** There is a noticeable drop in the customer count for the 50-65 age group compared to younger segments.
- **Support Calls:** Follows a right-skewed (decreasing) discrete distribution. The majority of customers make 0-3 calls, while very few reach the maximum of 10 calls. This is likely a key driver for Churn.
- **Payment Delay:** Distribution is uniform up to 20 days, followed by a sharp structural decline. Customers with >20 days delay are significantly fewer in number.
- **Total Spend:** There is a significantly higher concentration of customers spending between $500 - $1000 compared to the lower spending bracket ($100 - $500).
- **Last Interaction:** Shows a uniform engagement pattern for the first 15 days, which then drops to a lower, steady plateau for the remaining 16-30 days.


In [ ]:
numerical_features = [
    col for col in df.columns if df[col].dtype != 'category' and col != "Churn"
]

df[numerical_features].describe().round(2)

In [ ]:
sns.set_style("white")   # setup style

fig, axes = plt.subplots(nrows=2, ncols=4, figsize=(20, 10))
axes_list = axes.flatten()

for feature, ax in zip(numerical_features, axes_list):
    sns.histplot(data=df, x=feature,color = '#4C72B0' ,kde=True, ax=ax)
    ax.set_title(f"Distribution of {feature}")

axes_list[7].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=4, figsize=(20, 10))
axes_list = axes.flatten()

for feature, ax in zip(numerical_features, axes_list):
    sns.boxplot(data=df, x=feature, ax=ax)
    ax.set_title(f"Distribution of {feature}")

axes_list[7].axis("off")
plt.tight_layout()
plt.show()

#### **3.2. Category Features**

- **Gender**: The dataset has a slight imbalance between genders. Male customers account for 57%, while Female customers make up 43%.

- **Subscription Type**: Distribution is also balanced across all three tiers: Basic (32%), Premium (34%), and Standard (34%). 

- **Contract Length**: There is a notable gap in contract preferences. Both Annual and Quarterly contracts are dominant, each representing 40% of the user base. Interestingly, Monthly contracts are the least popular, accounting for only 20%.



In [ ]:
categorical_features = [col for col in df.columns if df[col].dtype == 'category']

sns.set_style("white")
sns.set_palette("pastel")

fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(20, 6))
axes_list = axes.flatten()

for feature, ax in zip(categorical_features, axes_list):
    # Draw countplot
    plot = sns.countplot(data=df, x=feature, ax=ax, color= "#9BC4DA")
    ax.set_title(f"Distribution of {feature}", fontsize=14, fontweight='bold')
     # Add percentage value for each bar
    total = len(df[feature].dropna())  # prevent divide for 0
    
    for p in plot.patches:
        if p.get_height() > 0:  
            height = p.get_height()
            pct = (height / total) * 100
            # Format number
            label = f"{pct:.0f}%" if pct >= 10 else f"{pct:.1f}%"
            ax.annotate(
                label,
                xy=(p.get_x() + p.get_width() / 2, height), # x,y positions
                xytext=(0, 3), # offset on 3 points
                textcoords="offset points",
                ha='center', va='bottom',
                fontsize=10, fontweight='bold'
            )
    # Gridline 
    ax.grid(axis='y', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

#### **4. Feature Relationship**

#### **4.1. Numerical and Target Variables**

##### **Important Feature**

- **Support Calls:** This is the most distinctive feature. The median for Churners is significantly higher (~5 calls) compared to Non-churners (~1 call). High interaction with support is a clear red flag.

- **Total Spend:** There is a stark contrast here. Customers who stay have a much higher median spend (~$750$) than those who churn (~$500$). Lower spending behavior is strongly correlated with attrition.

- **Payment Delay:** Churners show a higher median delay and a much wider interquartile range (IQR).

- **Last Interaction:** Customers who churn tend to have their last interaction longer ago (higher median) compared to loyal customers.

##### **Moderate Features**

- **Age:** Churners are generally older (median ~42) than those who stay (median ~37). While there is overlap, the shift in the distribution is statistically visible.

##### **Low Importance Features**

- **Tenure:** The boxplots for both groups are almost identical in terms of median (~30 months) and range. This suggests that how long a customer has been with the company does not directly influence their decision to leave in this specific dataset.
- **Usage Frequency:** Similar to Tenure, the distributions heavily overlap. The median usage is roughly the same for both groups (~15-17 times), making it a weak predictor for the model.


In [ ]:
plt.figure(figsize=(16, 12))
plt.suptitle("Relationship Between Numeric Features and Churn", fontsize=18, y=1.02)

for i, col in enumerate(numerical_features):
    plt.subplot(2, 4, i + 1)
    sns.boxplot(data=df, x="Churn", y=col, palette="pastel")

    plt.title(f"{col} vs. Churn")
    plt.xlabel("Churn (0 = No, 1 = Yes)")
    plt.ylabel(col)

plt.tight_layout(rect=(0, 0.03, 1, 0.98))
plt.show()

#### **Binning Age group and Churn**
- Group Aldult (25 - 39 years old) has the lowest churn rate, while Senior (> 60 years old) has 100% churn rate
- The two remaining group Young Aldult (18-24) and Mid-Career (40-59) has similar churn rate (~ 55%)


In [ ]:
# 1. Create bins
bins = [18, 24, 39, 59, df['Age'].max()]
labels = ['18-24', '25-39', '40-59', '> 60']

df['Age_Group'] = pd.cut(df['Age'], bins=bins, labels=labels, include_lowest=True)

# 2. Prepare data
age_churn_rate = df.groupby('Age_Group', observed=True)['Churn'].mean().reset_index()
age_churn_rate['Churn_Percentage'] = age_churn_rate['Churn'] * 100

age_dist = df['Age_Group'].value_counts().sort_index().reset_index()
age_dist.columns = ['Age_Group', 'Count']

# 3. Plot
sns.set_style("white")
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# --- Subplot 1: Distribution ---
sns.barplot(data=age_dist, x='Age_Group', y='Count', ax=axes[0], color="#a1c9f4")

axes[0].set_title('Age Group Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Age Group')
axes[0].set_ylabel('Count')


# --- Subplot 2: Churn Rate ---
sns.barplot(data=age_churn_rate, x='Age_Group', y='Churn_Percentage',
            ax=axes[1], color="#4d99d0")

axes[1].set_title('Churn Rate by Age Group', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Age Group')
axes[1].set_ylabel('Churn Rate (%)')

# Annotate %
for p in axes[1].patches:
    height = p.get_height()
    if height > 0:
        axes[1].annotate(f'{height:.1f}%',
                         (p.get_x() + p.get_width() / 2., height),
                         ha='center', va='bottom',
                         fontsize=9, fontweight='bold',
                         xytext=(0, 4),
                         textcoords='offset points')

plt.tight_layout()
plt.show()

#### **Last Interaction and Churn**

Last Interaction is binned into three groups: Highly Active (0-7 days), Active (7-15 days), and Dormant (> 15 days).

- Critical Threshold at 15 Days: Dormant, where Last Interaction exceeds 15 days, clearly shows a higher churn rate at ~ 66%
- For customers with a Last Interaction between 0 and 15 days, the churn rate remains perfectly stable at ~ *49%.


In [ ]:
# --- Binning ---
def classify_interaction_frequency(last_interaction):
    if 0 < last_interaction <= 7:
        return "Highly Active"
    elif 7 < last_interaction <= 15:
        return "Active"
    elif last_interaction > 15:
        return "Dormant"

df["Interaction Frequency"] = df["Last Interaction"].apply(
    classify_interaction_frequency
)

order = ["Highly Active", "Active", "Dormant"]

# --- Color mapping (consistent) ---
color_map = {
    "Highly Active": "#a1c9f4",  # light blue
    "Active": "#6fa8dc",         # medium blue
    "Dormant": "#1f4e79"         # dark blue (highlight risk)
}

colors = [color_map[o] for o in order]

# --- Churn rate ---
churn_rate_by_group = df.groupby("Interaction Frequency")["Churn"].mean().reindex(order).reset_index()
churn_rate_by_group["Churn Rate (%)"] = churn_rate_by_group["Churn"] * 100

# --- Distribution ---
dist = df["Interaction Frequency"].value_counts().reindex(order)

# --- Plot ---
sns.set_style("white")
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# --- Subplot 1: Pie chart ---
axes[0].pie(
    dist,
    labels=order,
    autopct='%1.1f%%',
    startangle=90,
    colors=colors
)
axes[0].set_title("Interaction Frequency Distribution", fontsize=13, fontweight='bold')

# --- Subplot 2: Bar chart ---
ax = sns.barplot(
    data=churn_rate_by_group,
    x="Interaction Frequency",
    y="Churn Rate (%)",
    ax=axes[1],
    palette=color_map,
    order=order
)

# Annotate %
for p in ax.patches:
    height = p.get_height()
    if height > 0:
        ax.annotate(f'{height:.1f}%',
                    (p.get_x() + p.get_width() / 2., height),
                    ha='center', va='bottom',
                    fontsize=11,
                    xytext=(0, 5),
                    textcoords='offset points')

axes[1].set_title("Churn Rate by Interaction Frequency", fontsize=13, fontweight='bold')
axes[1].set_ylabel("Churn Rate (%)")
axes[1].set_ylim(0, 100)

plt.tight_layout()
plt.show()

#### **4.2. Categorical and Target Variables**

- **Contract Length**: Customers with Contract Type Monthly have a 100% Churn Rate. Meanwhile, types Annual and 2.0 (Quarterly) have much lower and identical churn rates (~45%).  

- **Gender**: There is a visible difference between genders. Female has a higher churn rate (~65%) compared to Male (~50%). Gender carries a strong signal in this dataset and should be prioritized during feature selection.

- **Subscription Type**: The churn rate across all three subscription tiers is virtually identical (approx. 55-58% churn). Because the proportion of churn does not change regardless of whether a customer is on a Basic, Standard, or Premium plan, this feature provides zero discriminative power for the model.


In [ ]:
plt.figure(figsize=(18, 6))
plt.suptitle("Relationship Between Categorical Features and Churn", fontsize=18, y=1.03)

df_plot = df.copy()
df_plot["Churn"] = df_plot["Churn"].map({0.0: "No Churn", 1.0: "Churn"})

for i, col in enumerate(categorical_features):
    plt.subplot(1, 3, i + 1)

    sns.histplot(
        data=df_plot,
        x=col,
        hue="Churn", 
        multiple="fill", 
        stat="proportion", 
        discrete=True,  
        shrink=0.8,  
    )

    plt.title(f"Churn Rate by {col}")
    plt.xlabel(col)
    plt.ylabel("Proportion of Customers")

    if i == 0:
        sns.move_legend(plt.gca(), "upper left", bbox_to_anchor=(-0.25, 1))
    else:
        plt.gca().get_legend().remove()

plt.tight_layout(rect=(0, 0.03, 1, 0.98))
plt.show()

#### **4.3. Correlation Heatmap**

In [ ]:
# Define explicit order for ordinal features
subscription_order = [["Basic", "Standard", "Premium"]]
contract_order = [["Monthly", "Quarterly", "Annual"]]

# Initialize encoders with defined categories
subscription_encoder = OrdinalEncoder(categories=subscription_order)
contract_encoder = OrdinalEncoder(categories=contract_order)

# Apply encoding (use DataFrame format to keep 2D shape)
df["Subscription Type"] = subscription_encoder.fit_transform(
    df[["Subscription Type"]]
)

df["Contract Length"] = contract_encoder.fit_transform(
    df[["Contract Length"]]
)

# Label encoding for binary categorical feature (no reshape needed)
label_encoder = LabelEncoder()
df["Gender"] = label_encoder.fit_transform(df["Gender"])

df.head()

In [ ]:
# Ensuring all feature are numeric
numeric_df = df.select_dtypes(include=["float64", "int64",'int32'])
numeric_df.columns


In [ ]:
corr_matrix = numeric_df.corr()

plt.figure(figsize=(12, 9))
sns.heatmap(
    corr_matrix,
    annot=True,  # Display the correlation values on the heatmap
    fmt=".2f",  # Format the values to two decimal places
    cmap="coolwarm",  # Use a diverging colormap (blue=negative, red=positive)
    linewidths=0.5,  # Add fine lines between cells
    vmin=-1,  # Set the min of the colormap to -1
    vmax=1,  # Set the max of the colormap to +1
)

plt.title("Correlation Heatmap of Numeric Features", fontsize=16)
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

#### **Conclusion**
- `Support Calls` and `Total Spend` have the highest correlation with Churn, while features like `Tenure`, `Usage Frequency`, `Subscription Type`, `Contract Length` show nearly no linear relationship. 

- For the case of `Contract Length`, enventhough the Monthly Contract (only account for 20% dataset) show the 100% churn rate, the other two types (Annual and Quarterly) have nearly identical churn rate. Additionally, because Monthly contracts result in a 100% churn rate, the target variable becomes a constant within that specific group (zero variance). Binary feature like `is_monthly_contract` will also lead to near-zero results in Pearson correlation matrices. 

- As for final selected features, we will only choose `Age`, `Gender`, `Support Calls`, `Payment Delay`, `Total Spend` and `Last Interaction`. 